# 7.8 Feature Scaling, Initialization, and Stabilizing Training

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook covers the practical levers that make training numerically stable enough to trust: scaling, initialization, gradient control, and scheduling.


## 7.8.1 Why scaling matters

### Why It Matters
A network trained on features that live on wildly different scales can struggle to optimize. Standardizing the same data often leads to more stable updates.


In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(29)
X_np = np.column_stack([rng.normal(0, 1, 200), rng.normal(0, 1000, 200)])
y_np = (X_np[:, 0] + 0.001 * X_np[:, 1] > 0).astype(np.float32).reshape(-1, 1)
scaler = StandardScaler().fit(X_np)
X_scaled = scaler.transform(X_np)

def train_loss(X_arr):
    X = torch.tensor(X_arr, dtype=torch.float32)
    y = torch.tensor(y_np, dtype=torch.float32)
    model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    loss_fn = nn.BCELoss()
    for _ in range(120):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    return round(float(loss.item()), 4)

{"unscaled_final_loss": train_loss(X_np), "scaled_final_loss": train_loss(X_scaled)}


### Interpretation
Scaling does not change the information content, but it changes how easy that information is for gradient-based optimization to use.


## 7.8.2 Parameter initialization

### Why It Matters
Initialization determines the network’s starting point. Different schemes can change activation magnitudes before training even begins.


In [ ]:
import torch
from torch import nn

x = torch.randn(64, 20)
layer_default = nn.Linear(20, 50)
layer_xavier = nn.Linear(20, 50)
nn.init.xavier_uniform_(layer_xavier.weight)
nn.init.zeros_(layer_xavier.bias)

default_std = layer_default(x).std().item()
xavier_std = layer_xavier(x).std().item()
{"default_output_std": round(float(default_std), 4), "xavier_output_std": round(float(xavier_std), 4)}


### Interpretation
Reasonable initialization helps activations stay in a learnable range instead of collapsing too early.


## 7.8.3 Gradient stability and clipping

### Why It Matters
When gradients grow too large, clipping can keep the update numerically manageable.


In [ ]:
import torch
from torch import nn

model = nn.Sequential(nn.Linear(10, 40), nn.ReLU(), nn.Linear(40, 1))
X = 50 * torch.randn(32, 10)
y = 50 * torch.randn(32, 1)
loss = nn.MSELoss()(model(X), y)
loss.backward()
pre_clip = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters())).item()
nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
post_clip = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters())).item()
{"pre_clip_norm": round(float(pre_clip), 3), "post_clip_norm": round(float(post_clip), 3)}


### Interpretation
Clipping does not solve a bad model by itself, but it can prevent one unstable step from derailing training.


## 7.8.4 Dead ReLUs

### Why It Matters
ReLU units output zero for negative inputs. If a unit stays there for too many examples, it can stop contributing useful signal.


In [ ]:
import torch
from torch import nn

layer = nn.Linear(3, 4)
with torch.no_grad():
    layer.bias.fill_(-2.0)

batch = torch.tensor([[0.1, 0.2, 0.1], [0.0, 0.1, -0.2], [0.3, -0.1, 0.0]])
pre_activation = layer(batch)
post_activation = torch.relu(pre_activation)

{
    "mean_pre_activation": round(float(pre_activation.mean().item()), 3),
    "zero_fraction_after_relu": round(float((post_activation == 0).float().mean().item()), 3),
    "post_activation_sample": post_activation[0].round(decimals=3).tolist(),
}


### Interpretation
This is why initialization, learning rate, and input scaling all interact with activation behavior.


## 7.8.5 Learning-rate scheduling

### Why It Matters
A scheduler changes the learning rate over time. Here a simple step schedule reduces it after fixed intervals.


In [ ]:
import torch
from torch import nn

model = nn.Linear(3, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
lrs = []
dummy_x = torch.randn(4, 3)
dummy_y = torch.randn(4, 1)
loss_fn = nn.MSELoss()
for _ in range(6):
    lrs.append(round(opt.param_groups[0]['lr'], 4))
    opt.zero_grad()
    loss = loss_fn(model(dummy_x), dummy_y)
    loss.backward()
    opt.step()
    scheduler.step()
lrs


### Interpretation
Scheduling is one of the simplest ways to start with fast learning and finish with finer updates.


## 7.8.6 Unstable versus stabilized training

### Why It Matters
This final subsection compares a bad setup and a better setup on the same data by changing both scaling and optimizer settings.


In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(30)
X_np = np.column_stack([rng.normal(0, 1, 240), rng.normal(0, 500, 240), rng.normal(0, 0.2, 240)])
y_np = (0.8 * X_np[:, 0] - 0.002 * X_np[:, 1] + 1.2 * X_np[:, 2] > 0).astype(np.float32).reshape(-1, 1)
X_scaled = StandardScaler().fit_transform(X_np)

def fit(X_arr, lr):
    X = torch.tensor(X_arr, dtype=torch.float32)
    y = torch.tensor(y_np, dtype=torch.float32)
    model = nn.Sequential(nn.Linear(3, 12), nn.ReLU(), nn.Linear(12, 1), nn.Sigmoid())
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    losses = []
    for _ in range(40):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
        losses.append(round(float(loss.item()), 4))
    return losses[-5:]

{"unscaled_lr_0.2_tail": fit(X_np, 0.2), "scaled_lr_0.05_tail": fit(X_scaled, 0.05)}


### Interpretation
Stability is not luck. It usually reflects a stack of reasonable choices made together.
